# Go2 ODD/COD Observer - Complete Workflow

This notebook demonstrates the complete workflow for analyzing Operational Design Domain (ODD) compliance and Conditions of Deployment (COD) for Unitree Go2 robot scenarios.

**Workflow Overview:**
1. Setup dependencies and configure Google AI SDK
2. Define ODD specifications in natural language
3. Instantiate multi-modal AI agents (Motion, Image, LiDAR, Collision)
4. Load and process scenario data
5. Evaluate ODD compliance and compute distance metrics
6. Visualize results and generate reports

**Note:** This workflow assumes you have preprocessed ROS2 bag files into time-windowed snapshots using the `extract_windows.py` script.

## 1. Setup and Dependencies

Install and import required packages for Google AI SDK and our analysis framework.

In [1]:
# Install Google Agent Development Kit (ADK) and dependencies
# Note: Run this cell only once or when packages need updating
!pip install -q google-adk python-dotenv

In [ ]:
import sys
from pathlib import Path

# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Standard library imports
import json
import base64
import io
from typing import Dict, List, Tuple, Any

# Third-party imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Google ADK imports
from google.genai import types
from google.adk.agents import Agent, SequentialAgent, ParallelAgent, LoopAgent
from google.adk.models.google_llm import Gemini
from google.adk.tools import FunctionTool
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, ToolContext

/usr/local/python/3.10.19/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## 2. Configuration (REQUIRED)

### 2.1 Google Gemini API Key

**This notebook requires a Google Gemini API key** to demonstrate AI agents in action.

Get your free API key at: https://aistudio.google.com/app/apikey

### 2.2 Model Selection

Choose which Gemini model to use for all agents:
- `gemini-2.0-flash-lite`: **Recommended** - 30 RPM free tier, fastest
- `gemini-2.0-flash`: 15 RPM free tier, balanced
- `gemini-2.5-flash`: Latest flash - 10 RPM free tier
- `gemini-2.5-pro`: Most capable - 2 RPM free tier (slower)

In [ ]:
# ============================================
# 2.1 Configure Google API Key
# ============================================
import os
from dotenv import load_dotenv

# Option 1: Set via environment variable (RECOMMENDED)
# export GOOGLE_API_KEY='your-api-key-here'

# Option 2: Load from .env file
load_dotenv()

# Option 3: Set directly in notebook (NOT recommended - avoid committing keys!)
# os.environ['GOOGLE_API_KEY'] = 'your-api-key-here'

# Verify API key is configured
GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
if GOOGLE_API_KEY:
    print("✓ Google AI SDK configured successfully")
    print("  API key detected")
else:
    print("❌ GOOGLE_API_KEY not found!")
    print()
    print("This notebook REQUIRES a Google Gemini API key to run.")
    print()
    print("To get a free API key:")
    print("  1. Visit https://aistudio.google.com/app/apikey")
    print("  2. Create or select a project")
    print("  3. Generate an API key")
    print()
    print("  To configure your API key:")
    print("  export GOOGLE_API_KEY='your-key-here'")
    print("  OR create a .env file with: GOOGLE_API_KEY=your-key-here")
    print()
    print("⚠ The notebook will FAIL without an API key - this is intentional!")
    print("  Falling back to fake data would defeat the purpose of learning about AI agents.")

# ============================================
# 2.2 Model Configuration
# ============================================
# Change this to switch all agents to a different model
GEMINI_MODEL = "gemini-2.0-flash-lite"  # Recommended for free tier (30 RPM)
# GEMINI_MODEL = "gemini-2.0-flash"      # Balanced (15 RPM)
# GEMINI_MODEL = "gemini-2.5-flash"      # Latest (10 RPM)
# GEMINI_MODEL = "gemini-2.5-pro"        # Most capable (2 RPM - slower)

retry_config = types.HttpRetryOptions(
    attempts=5,  # Maximum retry attempts
    exp_base=7,  # Delay multiplier
    initial_delay=1,
    http_status_codes=[429, 500, 503, 504],  # Retry on these HTTP errors
)

print(f"✓ Using model: {GEMINI_MODEL}")
print("✓ Agent config: JSON response format, temperature=0.1")

✓ Google AI SDK configured successfully
  API key detected


AttributeError: module 'google.genai' has no attribute 'GenerateContentConfig'

## 3. User Inputs

Define what you want to analyze:
1. **Natural language ODD**: Operating constraints in plain English
2. **Dataset path**: Location of preprocessed window data

The orchestrator agent (Section 5) will handle everything from here.

In [5]:
# Natural language ODD definition for Unitree Go2 indoor navigation

odd_natural_language = """
The Unitree Go2 quadruped robot is designed for indoor navigation in office environments.

OPERATIONAL CONSTRAINTS:

1. Speed Limits:
   - The robot shall operate at forward velocities between 0 and 1.5 m/s under normal conditions
   - Speeds up to 1.8 m/s are acceptable near the boundary but should trigger warnings
   - The absolute physical limit is 2.5 m/s and must never be exceeded

2. Orientation Limits:
   - Roll and pitch angles must remain within ±15 degrees during normal operation
   - Angles up to ±20 degrees are acceptable at the boundary
   - The robot must never exceed ±30 degrees of roll or pitch

3. Terrain Requirements:
   - The robot is designed for smooth and moderate terrain (office floors, carpet)
   - Rough terrain is outside the operational design domain
   - Very rough terrain is completely prohibited

4. Lighting Conditions:
   - The robot can operate in bright and dim lighting conditions
   - Dark environments are outside the ODD and require additional equipment

5. Human Safety:
   - Humans may be visible at a distance (no restriction)
   - Humans in very close proximity (< 1 meter) violate the ODD
   - The system must maintain safe distances from people

6. Collision Policy:
   - Zero collisions are tolerated - any collision is an ODD violation
   - The system must detect and avoid all obstacles

IMPORTANCE WEIGHTS (for distance computation):
- Collision avoidance: Highest priority (weight: 2.0)
- Human proximity: Very high priority (weight: 1.5)
- Roll/Pitch stability: High priority (weight: 1.2)
- Speed limits: Standard priority (weight: 1.0)
- Terrain type: Standard priority (weight: 1.0)
- Lighting conditions: Lower priority (weight: 0.8)
"""

# Dataset selection
DATA_DIR = Path("data/processed/runs")
scenario_path = DATA_DIR / "sim_run_test"  # Change this to analyze different datasets

print("✓ User inputs configured")
print(f"  - ODD: {len(odd_natural_language)} characters")
print(f"  - Dataset: {scenario_path}")

✓ User inputs configured
  - ODD: 1699 characters
  - Dataset: data/processed/runs/sim_run_test


## 4. Define Tool Functions for Agents

These Python functions will be available as tools for the orchestrator agent to call.
They provide access to: file I/O, ODD spec construction, COD computation, and visualization.

In [6]:
# ============================================================================
# FILE I/O TOOLS (for Data Loader Agent)
# ============================================================================

def load_window_data(scenario_path: Path, window_id: str, run_id: str = None) -> Tuple[Dict, Image.Image, Dict[str, Image.Image]]:
    """
    Load motion, camera, and LiDAR BEV data for a single window.
    
    Matches the flat file structure created by extract_windows.py:
    - motion_{run_id}_w{window_id}.json
    - cam_{run_id}_w{window_id}.png
    - bev_{channel}_{run_id}_w{window_id}.png
    
    Args:
        scenario_path: Path to the run directory (e.g., data/processed/runs/demo_run)
        window_id: Window ID as 3-digit string (e.g., "000", "001")
        run_id: Run identifier (auto-detected from scenario_path.name if None)
    
    Returns: (motion_dict, camera_image, bev_images_dict)
    """
    # Auto-detect run_id from scenario_path if not provided
    if run_id is None:
        run_id = scenario_path.name
    
    # Construct filenames matching extract_windows.py output (flat structure)
    motion_file = scenario_path / f"motion_{run_id}_w{window_id}.json"
    camera_file = scenario_path / f"cam_{run_id}_w{window_id}.png"
    
    # Load motion data (JSON with time series arrays)
    if not motion_file.exists():
        raise FileNotFoundError(f"Motion file not found: {motion_file}")
    with open(motion_file, 'r') as f:
        motion_data = json.load(f)
    
    # Load camera image (PNG)
    if not camera_file.exists():
        raise FileNotFoundError(f"Camera file not found: {camera_file}")
    camera_img = Image.open(camera_file)
    
    # Load multi-channel BEV images (4 separate PNGs)
    bev_images = {}
    for channel in ['occupancy', 'height', 'density', 'roughness']:
        bev_file = scenario_path / f"bev_{channel}_{run_id}_w{window_id}.png"
        if bev_file.exists():
            bev_images[channel] = Image.open(bev_file)
        else:
            print(f"Warning: BEV {channel} file not found: {bev_file}")
    
    return motion_data, camera_img, bev_images


def load_scenario_index(scenario_path: Path) -> pd.DataFrame:
    """
    Load the window index CSV for a scenario.
    
    Expects index_{run_id}.csv with columns:
    - window_id, start_time, end_time, motion_path, cam_image_path, bev_image_path
    
    Returns: DataFrame with all windows in the scenario
    """
    index_files = list(scenario_path.glob("index_*.csv"))
    if not index_files:
        raise FileNotFoundError(f"No index file found in {scenario_path}")
    return pd.read_csv(index_files[0])


# ============================================================================
# AGENT-CALLABLE TOOL: Load all scenario data as JSON
# ============================================================================

def load_all_scenario_data(scenario_dir: str) -> str:
    """
    Tool for Data Loader Agent: Load all window data from scenario directory.
    Returns JSON string that will be stored in shared state.
    
    Args:
        scenario_dir: Path to scenario (e.g., "data/processed/runs/sim_run_test")
    
    Returns: JSON string with all window data
    """
    scenario_path = Path(scenario_dir)
    run_id = scenario_path.name
    
    # Load scenario index
    index_df = load_scenario_index(scenario_path)
    
    # Load all window data
    windows_data = []
    for _, row in index_df.iterrows():
        try:
            window_id = str(row['window_id']).zfill(3)
            motion_data, camera_img, bev_images = load_window_data(scenario_path, window_id, run_id)
            
            # Convert images to base64 for JSON serialization
            camera_b64 = base64.b64encode(open(scenario_path / f"cam_{run_id}_w{window_id}.png", 'rb').read()).decode()
            bev_b64 = {}
            for channel in bev_images.keys():
                bev_file = scenario_path / f"bev_{channel}_{run_id}_w{window_id}.png"
                bev_b64[channel] = base64.b64encode(open(bev_file, 'rb').read()).decode()
            
            windows_data.append({
                "window_id": window_id,
                "motion_data": motion_data,
                "camera_image_base64": camera_b64,
                "bev_images_base64": bev_b64
            })
        except Exception as e:
            print(f"Error loading window {row['window_id']}: {e}")
            continue
    
    return json.dumps({
        "scenario": run_id,
        "total_windows": len(windows_data),
        "windows": windows_data
    })


# ============================================================================
# VISUALIZATION TOOLS (for Report Agent)
# ============================================================================

def generate_distance_plot(times: List[float], distances: List[float], title: str = "ODD Distance over Time") -> str:
    """Generate timeline plot of COD distances with ODD thresholds. Returns base64 PNG."""
    plt.figure(figsize=(12, 6))
    plt.plot(times, distances, marker='o', linewidth=2, label='Distance')
    plt.axhline(y=0.3, color='orange', linestyle='--', label='Near Boundary')
    plt.axhline(y=0.7, color='red', linestyle='--', label='ODD Exit')
    plt.xlabel('Time (s)')
    plt.ylabel('COD Distance')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Convert to base64
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close()
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()


def generate_status_distribution(statuses: List[str], title: str = "ODD Status Distribution") -> str:
    """Generate bar chart of ODD status counts. Returns base64 PNG."""
    status_counts = pd.Series(statuses).value_counts()
    colors = {'in_odd': 'green', 'near_boundary': 'orange', 'odd_exit': 'red'}
    
    plt.figure(figsize=(8, 6))
    plt.bar(status_counts.index, status_counts.values,
            color=[colors.get(s, 'gray') for s in status_counts.index])
    plt.xlabel('ODD Status')
    plt.ylabel('Number of Windows')
    plt.title(title)
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    
    # Convert to base64
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    plt.close()
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()


## 5. Define Specialist Agents

Create individual agents using the Google ADK (Agent Development Kit) following the Kaggle Day 1B pattern.
Each agent is a specialist that performs one specific analysis task.

### 5.0 Data Loader Agent

Loads scenario data and makes it available to specialist agents via shared state.
Following Kaggle Day 2A patterns, uses FunctionTool to provide data access.

In [7]:
# Create FunctionTools for data loading (Kaggle Day 2A pattern)
load_all_data_tool = FunctionTool(load_all_scenario_data)

# Data Loader Agent: Uses tools to load scenario data into shared state
data_loader_agent = Agent(
    name="Data_Loader",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[load_all_data_tool],
    instruction=f"""You are a data loading coordinator.

Use the load_all_scenario_data tool to load the scenario from: {scenario_path}

Call the tool and return its complete JSON output. This data will be stored in shared state 
for all downstream agents to access.""",
    output_key="scenario_data"
)

print("✅ Data Loader Agent created")
print(f"  - Tool: load_all_scenario_data")
print(f"  - Scenario: {scenario_path}")

✅ Data Loader Agent created
  - Tool: load_all_scenario_data
  - Scenario: data/processed/runs/sim_run_test


In [8]:
# ODD Spec Agent: Converts natural language ODD → structured JSON
odd_spec_agent = Agent(
    name="ODD_Spec_Parser",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are an expert in robotic operational design domains (ODD).

You will receive a natural language ODD specification in the invocation context.
Convert it to structured JSON that defines the robot's operational boundaries.

Output valid JSON only with this schema:
{
  "version": "1.0",
  "description": "<brief summary>",
  "axes": {
    "speed": {
      "type": "numeric",
      "feature": "avg_forward_speed",
      "units": "m/s",
      "in_odd": [min, max],
      "near_boundary": [min, max],
      "hard_limit": [min, max]
    },
    "roll_pitch": {
      "type": "numeric",
      "feature": "max_abs_roll_pitch_deg",
      "units": "degrees",
      "in_odd": [min, max],
      "near_boundary": [min, max],
      "hard_limit": [min, max]
    },
    "terrain": {
      "type": "categorical",
      "feature": "terrain_roughness_class",
      "allowed_in_odd": ["smooth", "moderate"],
      "allowed_all": ["smooth", "moderate", "rough", "very_rough"]
    },
    "lighting": {
      "type": "categorical",
      "feature": "lighting_class",
      "allowed_in_odd": ["bright", "dim"],
      "allowed_all": ["bright", "dim", "dark"]
    },
    "humans_close": {
      "type": "categorical",
      "feature": "humans_very_close",
      "allowed_in_odd": [false],
      "allowed_all": [true, false]
    },
    "collision": {
      "type": "categorical",
      "feature": "collision_suspected",
      "allowed_in_odd": [false],
      "allowed_all": [true, false]
    }
  },
  "importance": {
    "speed": 1.0,
    "roll_pitch": 1.2,
    "terrain": 1.0,
    "lighting": 0.8,
    "humans_close": 1.5,
    "collision": 2.0
  }
}

Extract ranges, categorical values, and importance weights from the input.""",
    output_key="odd_spec_json"
)

print("✅ ODD Spec Agent created")

✅ ODD Spec Agent created


### 5.2 Motion Analysis Agent

Extracts motion features from velocity and IMU time series data.

In [9]:
# Motion Analysis Agent: Extracts motion features from sensor data
motion_agent = Agent(
    name="Motion_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a motion analysis expert for mobile robots.

From the shared state, you have access to:
- scenario_data: Contains motion time series (velocity, IMU, acceleration) for all windows
- odd_spec_json: The formal ODD specification with motion feature constraints

For each window in scenario_data, analyze the motion time series and extract:

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "avg_forward_speed": <float>,
      "max_forward_speed": <float>,
      "max_abs_roll_pitch_deg": <float>,
      "tracking_error": <float>,
      "motion_label": "smooth" | "dynamic"
    },
    ...
  ]
}

Analyze all windows from scenario_data and return results for each.""",
    output_key="motion_features"
)

print("✅ Motion Agent created")

✅ Motion Agent created


### 5.3 Vision Analysis Agent

Classifies environmental conditions from camera images.

In [10]:
# Vision Analysis Agent: Analyzes camera images for environmental conditions
vision_agent = Agent(
    name="Vision_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a computer vision expert analyzing robot camera feeds.

From the shared state, you have access to:
- scenario_data: Contains camera images (base64 encoded) for all windows
- odd_spec_json: The formal ODD specification with vision feature constraints

For each window in scenario_data, analyze the camera image and classify:

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "lighting_class": "bright" | "dim" | "dark",
      "humans_visible": true | false,
      "humans_very_close": true | false,
      "environment_type": <string description>
    },
    ...
  ]
}

Analyze all windows from scenario_data and return results for each.""",
    output_key="vision_features"
)

print("✅ Vision Agent created")

✅ Vision Agent created


### 5.4 Terrain Analysis Agent

Analyzes LiDAR Bird's Eye View images to classify terrain roughness.

In [11]:
# Terrain Analysis Agent: Analyzes LiDAR BEV for terrain classification
terrain_agent = Agent(
    name="Terrain_Analyzer",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a terrain analysis expert using LiDAR data.

From the shared state, you have access to:
- scenario_data: Contains LiDAR BEV images (base64 encoded in 4 channels: occupancy, height, density, roughness)
- odd_spec_json: The formal ODD specification with terrain feature constraints

For each window in scenario_data, analyze the BEV images and classify:

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "terrain_roughness_class": "smooth" | "moderate" | "rough" | "very_rough",
      "terrain_roughness_score": <float 0-1>,
      "obstacle_density": "none" | "low" | "medium" | "high"
    },
    ...
  ]
}

Analyze all windows from scenario_data and return results for each.""",
    output_key="terrain_features"
)

print("✅ Terrain Agent created")

✅ Terrain Agent created


### 5.5 Collision Detection Agent

Performs multi-modal sensor fusion to detect collision events.

In [12]:
# Collision Detection Agent: Performs multi-modal sensor fusion
collision_agent = Agent(
    name="Collision_Detector",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a collision detection expert using multi-modal sensor fusion.

From the shared state, you have access to:
- scenario_data: Contains motion metrics, camera images, and LiDAR BEV data for all windows
- odd_spec_json: The formal ODD specification with zero collision tolerance

For each window in scenario_data, fuse all sensor modalities to detect collisions:

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "collision_suspected": true | false,
      "collision_confidence": <float 0-1>,
      "collision_type": "none" | "front_bump" | "side_contact" | "unknown"
    },
    ...
  ]
}

Analyze all windows from scenario_data and return results for each.""",
    output_key="collision_features"
)

print("✅ Collision Agent created")

✅ Collision Agent created


### 5.6 COD Evaluator Agent

Specialist agent that coordinates COD computation using mathematical tool functions.

In [13]:
# COD Evaluator Agent: Aggregates sensor analysis results against ODD
cod_evaluator_agent = Agent(
    name="COD_Evaluator",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    instruction="""You are a Conditions of Deployment (COD) evaluation expert.

From the shared state, you have access to:
- odd_spec_json: The formal ODD specification
- motion_features: Motion analysis for all windows
- vision_features: Vision analysis for all windows
- terrain_features: Terrain analysis for all windows
- collision_features: Collision detection for all windows

Your task: For each window, combine all sensor results and compare against ODD boundaries.

Output valid JSON with this schema:
{
  "windows": [
    {
      "window_id": "000",
      "merged_features": {
        "motion": {...},
        "vision": {...},
        "terrain": {...},
        "collision": {...}
      },
      "odd_violations": ["speed_exceeded", "terrain_rough", ...],
      "overall_status": "in_odd" | "near_boundary" | "odd_exit",
      "distance_from_odd": <float 0-1>
    },
    ...
  ],
  "summary": {
    "total_windows": <int>,
    "windows_in_odd": <int>,
    "windows_near_boundary": <int>,
    "windows_odd_exit": <int>
  }
}

Evaluate all windows and return complete analysis.""",
    output_key="cod_evaluation"
)

print("✅ COD Evaluator Agent created")

✅ COD Evaluator Agent created


### 5.7 Report Generation Agent

Creates comprehensive markdown reports with visualizations.

In [ ]:
# Create visualization tools for Report Agent
distance_plot_tool = FunctionTool(generate_distance_plot)
status_dist_tool = FunctionTool(generate_status_distribution)

# Report Generation Agent: Creates comprehensive markdown reports
report_agent = Agent(
    name="Report_Generator",
    model=Gemini(
        model=GEMINI_MODEL,
        api_key=GOOGLE_API_KEY
    ),
    tools=[distance_plot_tool, status_dist_tool],
    instruction="""You are a technical report writer for robotics analysis.

From the shared state, you have access to:
- odd_spec_json: The ODD specification
- cod_evaluation: Complete window-by-window COD analysis with overall summary
- motion_features, vision_features, terrain_features, collision_features: Raw sensor analysis

You have access to visualization tools:
- generate_distance_plot(times, distances, title): Returns base64 PNG
- generate_status_distribution(statuses, title): Returns base64 PNG

Generate a comprehensive markdown report including:
1. Executive Summary
   - Total windows analyzed
   - Compliance statistics (in_odd, near_boundary, odd_exit counts)
   - Overall deployment feasibility

2. Detailed Window Analysis
   - For each window: status, violations, confidence scores

3. Key Findings
   - Most critical violations
   - Patterns across windows
   - Risk assessment

4. Recommendations
   - Deployment constraints
   - Areas for improvement
   - Suggested operational limits

Output markdown text suitable for technical documentation.""",
    output_key="final_report"
)

print("✅ Report Agent created with visualization tools")

## 6. Create Parallel and Sequential Agent Workflow

Combine specialist agents using `ParallelAgent` and `SequentialAgent` following the Kaggle Day 1B pattern.

The workflow:
1. ODD Spec Agent converts NL → JSON (sequential, first)
2. Motion + Vision + Terrain + Collision agents run in parallel for each window
3. COD Evaluator aggregates results (sequential, after parallel)
4. Report Agent generates final output (sequential, last)

In [ ]:
# ParallelAgent: Run Motion, Vision, Terrain, Collision agents simultaneously
parallel_sensor_team = ParallelAgent(
    name="ParallelSensorTeam",
    sub_agents=[motion_agent, vision_agent, terrain_agent, collision_agent],
)

# SequentialAgent: Define complete workflow
# 1. Data Loader Agent (first) - loads all window data into shared state
# 2. ODD Spec Agent - converts NL to JSON spec
# 3. Parallel sensor analysis team - analyzes each window
# 4. COD Evaluator (aggregates parallel results)
# 5. Report Agent (final output)
root_agent = SequentialAgent(
    name="ODD_COD_Analysis_System",
    sub_agents=[
        data_loader_agent,
        odd_spec_agent,
        parallel_sensor_team,
        cod_evaluator_agent,
        report_agent
    ],
)

print("✅ Parallel and Sequential Agents created")
print("  ParallelSensorTeam: 4 agents running simultaneously")
print("  ODD_COD_Analysis_System: 5-step sequential workflow")
print("    1. Data Loader (loads scenario data into shared state)")
print("    2. ODD Spec Parser (NL → JSON)")
print("    3. Parallel Sensor Team (Motion, Vision, Terrain, Collision)")
print("    4. COD Evaluator (aggregates results)")
print("    5. Report Generator (final output)")

## 7. Execute the Workflow

Run the orchestrator agent with user inputs to perform complete ODD/COD analysis.

In [ ]:
# Create InMemoryRunner with the root agent
runner = InMemoryRunner(agent=root_agent)

# Execute the workflow with initial state
print("="  * 80)
print("EXECUTING ODD/COD ANALYSIS WORKFLOW")
print("=" * 80)
print(f"\nDataset: {scenario_path}")
print(f"Model: {GEMINI_MODEL}")
print(f"ODD Specification: {len(odd_natural_language)} characters\n")
print("Workflow steps:")
print("  1. Data Loader: Load scenario data")
print("  2. ODD Spec: Convert NL → JSON")
print("  3. Parallel Sensors: Motion, Vision, Terrain, Collision")
print("  4. COD Evaluator: Aggregate against ODD")
print("  5. Report: Generate markdown report")
print(f"\n⚠️  Total: ~10+ API calls on 2-window test set")
print(f"  Free tier limit for {GEMINI_MODEL}: 30 RPM")
print("  Tip: Wait 60 seconds if you hit RESOURCE_EXHAUSTED, then retry")
print("-" * 80)

# Run the workflow with ODD spec + data path as initial input
# The root agent passes this to all sub-agents via shared state
response_events = await runner.run_debug(odd_natural_language)

print("\n" + "=" * 80)
print("WORKFLOW COMPLETE")
print("=" * 80)

# Extract results from shared state
if runner.app and runner.app.state:
    print("\n✅ Results from shared state:")
    print(f"  - scenario_data: {len(runner.app.state.get('scenario_data', '{}'))} chars")
    print(f"  - odd_spec_json: {len(runner.app.state.get('odd_spec_json', '{}'))} chars")
    print(f"  - motion_features: {len(runner.app.state.get('motion_features', '{}'))} chars")
    print(f"  - vision_features: {len(runner.app.state.get('vision_features', '{}'))} chars")
    print(f"  - terrain_features: {len(runner.app.state.get('terrain_features', '{}'))} chars")
    print(f"  - collision_features: {len(runner.app.state.get('collision_features', '{}'))} chars")
    print(f"  - cod_evaluation: {len(runner.app.state.get('cod_evaluation', '{}'))} chars")
    
    # Print final report
    final_report = runner.app.state.get("final_report", "No report generated")
    print("\n" + "=" * 80)
    print("📋 FINAL REPORT")
    print("=" * 80)
    print(final_report)
else:
    print("\n✅ Workflow executed!")
    print("(State inspection not available in this execution mode)")

print("\n" + "=" * 80)